### GPT2 Testing based on Huggingspace 

#### Huggingspace Transformers Package 

Reference : https://huggingface.co/docs/transformers/ko/index <br>
reference : https://huggingface.co/docs/transformers/installation

설치 방법 : 

pip install transformers
pip install hf_xet

In [6]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer

def generate_text(prompt: str):
  model = GPT2LMHeadModel.from_pretrained('gpt2')
  tokenizer = GPT2Tokenizer.from_pretrained('gpt2')

  # encoding : string -> token indices tensor
  encoded_input  = tokenizer.encode(prompt, return_tensors='pt')
  print(f"encoded_input : {encoded_input}")

  # text generation
  output = model.generate(encoded_input)

  # decoding : generated token indices -> string
  generated_text = tokenizer.decode(output[0], skip_special_tokens=False)

  return generated_text


# sample test code
prompt = "In the beginning, "
generated_text = generate_text(prompt)
print(generated_text)

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


tensor([[ 818,  262, 3726,   11,  220]])
In the beginning,  I was a little bit of a fan of the original series, but I was also a little


#### Iterative Text Generation

In [7]:
import torch
import torch.nn.functional as F
from transformers import GPT2LMHeadModel, GPT2Tokenizer

torch.manual_seed(42)

def generate_text(prompt: str, iterations: int = 50):
  model = GPT2LMHeadModel.from_pretrained('gpt2')
  tokenizer = GPT2Tokenizer.from_pretrained('gpt2')

  model.eval()

  # 초기 텍스트 설정
  generated_text = prompt

  for _ in range(iterations):
    # encoding : string -> token indices tensor
    input_ids = tokenizer.encode(generated_text, return_tensors='pt')

    # 모델 출력 : NamedTuple(logits, past_key_values, hidden_states:optional, attentions:optional)
    # logits : 현재 토큰 시퀀스에 대한 다음 토큰 예측 확률
    output = model(input_ids)

    # 생성된 마지막 예측 확률 계산
    probs = F.softmax(output.logits[:, -1, :], dim=-1)

    # 다음 토큰 샘플링
    next_token = torch.multinomial(probs, num_samples=1)

    # 생성된 토큰 추가
    generated_text += tokenizer.decode(next_token.item(), skip_special_tokens=False)
    print(generated_text)

  return generated_text


prompt = "In the beginning, "
generated_text = generate_text(prompt)
print(generated_text)

In the beginning,  
In the beginning,   Bak
In the beginning,   Bakunin
In the beginning,   Bakunin was
In the beginning,   Bakunin was not
In the beginning,   Bakunin was not interested
In the beginning,   Bakunin was not interested in
In the beginning,   Bakunin was not interested in revolution
In the beginning,   Bakunin was not interested in revolution.
In the beginning,   Bakunin was not interested in revolution.  
In the beginning,   Bakunin was not interested in revolution.   He
In the beginning,   Bakunin was not interested in revolution.   He approached
In the beginning,   Bakunin was not interested in revolution.   He approached the
In the beginning,   Bakunin was not interested in revolution.   He approached the project
In the beginning,   Bakunin was not interested in revolution.   He approached the project like
In the beginning,   Bakunin was not interested in revolution.   He approached the project like a
In the beginning,   Bakunin was not interested in revolution.   He 

#### GPT2 Fine Tunning 을 위한 Tiktokenizer 사용

GPT2Tokenizer 는 최대 1024 개의 token 만 변환할 수 있습니다. 

Fine Tunning 을 위해선 전체 데이터를 Token 화 하는 것이 편하기 때문에

따라서, 더 많은 token 을 encode, decode 할 수 있는 Tiktokenizer 를 사용합니다.

web : https://tiktokenizer.vercel.app/ <br>
github : https://github.com/dqbd <br>
pypi : https://pypi.org/project/tiktoken/

pip install tiktoken

In [8]:
import torch
import torch.nn.functional as F
from transformers import GPT2LMHeadModel
import tiktoken

torch.manual_seed(42)

def generate_text(prompt: str, iterations: int = 50):
  model = GPT2LMHeadModel.from_pretrained('gpt2')
  # tiktoken 을 사용하여 tokenizer 를 생성
  tokenizer = tiktoken.encoding_for_model('gpt2')

  model.eval()

  # 초기 텍스트 설정
  generated_text = prompt

  for _ in range(iterations):
    # encoding : string -> token indices tensor
    input_ids = torch.tensor(tokenizer.encode(generated_text)).unsqueeze(0)

    # 모델 출력 : NamedTuple(logits, past_key_values, hidden_states:optional, attentions:optional)
    # logits : 현재 토큰 시퀀스에 대한 다음 토큰 예측 확률
    output = model(input_ids)

    # 생성된 마지막 예측 확률 계산
    probs = F.softmax(output.logits[:, -1, :], dim=-1)

    # 다음 토큰 샘플링
    next_token = torch.multinomial(probs, num_samples=1)

    # 생성된 토큰 추가
    next_word = tokenizer.decode([next_token.item()])
    generated_text += next_word
    print(generated_text)

  return generated_text


prompt = "In the beginning, "
generated_text = generate_text(prompt)
print(generated_text)

In the beginning,  
In the beginning,   Bak
In the beginning,   Bakunin
In the beginning,   Bakunin was
In the beginning,   Bakunin was not
In the beginning,   Bakunin was not interested
In the beginning,   Bakunin was not interested in
In the beginning,   Bakunin was not interested in revolution
In the beginning,   Bakunin was not interested in revolution.
In the beginning,   Bakunin was not interested in revolution.  
In the beginning,   Bakunin was not interested in revolution.   He
In the beginning,   Bakunin was not interested in revolution.   He approached
In the beginning,   Bakunin was not interested in revolution.   He approached the
In the beginning,   Bakunin was not interested in revolution.   He approached the project
In the beginning,   Bakunin was not interested in revolution.   He approached the project like
In the beginning,   Bakunin was not interested in revolution.   He approached the project like a
In the beginning,   Bakunin was not interested in revolution.   He 

### Bible Data 를 이용한 GPT2 Fine Tunning

generate_text 함수를 일부 수정합니다.

1. model 을 외부에서 입력받도록 수정합니다.
2. model 이 cpu 에서 동작하도록 수정합니다.

In [10]:
def generate_text(prompt: str, model, iterations: int = 50):
  # model 을 cpu 에 적재
  model = model.to('cpu')

  # tiktoken 을 사용하여 tokenizer 를 생성
  tokenizer = tiktoken.encoding_for_model('gpt2')

  model.eval()

  # 초기 텍스트 설정
  generated_text = prompt

  for _ in range(iterations):
    # encoding : string -> token indices tensor
    input_ids = torch.tensor(tokenizer.encode(generated_text)).unsqueeze(0)

    # 모델 출력 : NamedTuple(logits, past_key_values, hidden_states:optional, attentions:optional)
    # logits : 현재 토큰 시퀀스에 대한 다음 토큰 예측 확률
    output = model(input_ids)

    # 생성된 마지막 예측 확률 계산
    probs = F.softmax(output.logits[:, -1, :], dim=-1)

    # 다음 토큰 샘플링
    next_token = torch.multinomial(probs, num_samples=1)

    # 생성된 토큰 추가
    next_word = tokenizer.decode([next_token.item()])
    generated_text += next_word

  return generated_text

Bible Data 를 로드하여 token 화 합니다.


In [12]:
with open('bible.txt') as f:
  # bible read
  text = f.read()  
  # tiktoken 을 사용하여 tokenizer 를 생성
  tokenizer = tiktoken.encoding_for_model('gpt2')
  # text -> tokens
  tokens = tokenizer.encode(text)
  

# tokens -> tensor
bible_tokens = input_ids = torch.tensor(tokens)
print(bible_tokens.shape)

torch.Size([1258315])


Get a random batch of the data 함수 구현

In [15]:
# block_size : 몇 개의 token 까지 관계를 볼 것인지를 나타냄.
def get_batch(data, batch_size, block_size):
  # randomly choose batch size indices 
  # (선택할 데이터 블록의 첫번째 인덱스를 batch size 만큼 랜던 선택)
  indices = torch.randint(len(data) - block_size, (batch_size,))
  # get a batch data
  batch_input = torch.stack([data[i:i+block_size] for i in indices])
  batch_output = torch.stack([data[i+1:i+block_size+1] for i in indices])
  return batch_input, batch_output


batch_input, batch_output = get_batch(bible_tokens, 1, 10)
print(batch_input)
print(batch_output)

tensor([[ 1179,   436,    13,   198, 12016, 38182,  8699,    25,  2857,   197]])
tensor([[  436,    13,   198, 12016, 38182,  8699,    25,  2857,   197,  1544]])


#### Training Steps

Torch Device 를 설정합니다.

In [ ]:
print(torch.__version__)

if torch.backends.mps.is_available():
  my_device = torch.device('mps')
elif torch.cuda.is_available():
  my_device = torch.device('cuda')
else:
  my_device = torch.device('cpu')

print(my_device)

2.7.0+cu126
cuda


In [18]:
import torch.nn as nn
from torch.optim import Adam

from transformers import GPT2LMHeadModel

torch.manual_seed(42)

# training parameters
learning_rate = 1e-5
steps = 1000000


# model creation
model = GPT2LMHeadModel.from_pretrained('gpt2')
model.to(my_device)

# optimizer
optimizer = Adam(model.parameters(), lr=learning_rate)
# loss function
criterion = nn.CrossEntropyLoss()

batch_size = 4
block_size = 1024
total_loss = 0

for step in range(steps):
  # get a random batch data
  batch_input, batch_output = get_batch(bible_tokens, batch_size, block_size)
  # to device tensor
  batch_input = batch_input.to(my_device)
  batch_output = batch_output.to(my_device)

  # forward pass
  # model output : NamedTuple(logits, past_key_values, hidden_states:optional, attentions:optional)
  # logits : 현재 토큰 시퀀스에 대한 다음 토큰 예측 확률
  logits = model(batch_input).logits

  # view(-1, logits.size(-1)) : 2D (batch_size * block_size, voca_size)
  # voca_size 는 각각 character probability 를 나타냄.
  logits = logits.view(-1, logits.size(-1))
  # view(-1) : 1D (batch_size * block_size,)
  batch_output = batch_output.view(-1)

  loss = criterion(logits, batch_output)
  total_loss += loss.item()

  # zero the gradients
  optimizer.zero_grad()
  # backward pass : compute gradient of the loss with respect to model parameters
  loss.backward()
  # step function : update the weights
  optimizer.step()

  # 10 step 마다 문장 생성 결과 출력
  if step % 10 == 0:
    print(f"step: {step}, loss: {loss.item()}")

    prompt_text = "In the beginning,"
    print(generate_text(prompt_text, model, 50))

    # generate_text 함수 안에서 cpu device, eval mode 로 변경하기 때문에
    # my_device, train mode 로 다시 변경해야 함.
    model.to(my_device)
    model.train()


step: 0, loss: 3.2169764041900635
In the beginning, states created the US Unified Features — requirements that defined less oppressive political systems. Now it's all about the feds controlling the first 2 percent of the population. These people can't breathe, start voting, limit political freedom at a young age, and think
step: 10, loss: 2.9722609519958496
In the beginning, there was lots of allegations I never thought I would get and a lot of anxiety.


Will Oni Rosen and several outdoorsmen filed a lawsuit this year to stop CGG for their naked photos showing religious


creator of the religion
step: 20, loss: 2.9022743701934814
In the beginning, the problem that surrounded the application of a Stewart rule to Unix sites started as a mystery with Mac OS X, before it entered mainstream use around 1970. Stewart brought rip-routing, operators, RAID, and shared data, but followed several of the
step: 30, loss: 2.692368984222412
In the beginning, they all worked, on rocks come and go. The

KeyboardInterrupt: 